# Official HEC-HMS Guide Mirror: Calibration and Validation

Official guide: https://www.hec.usace.army.mil/confluence/hmsdocs/hmsguides/model-calibration-and-validation

This notebook maps the official calibration and validation workflow to hms-commander primitives: identify observed data, inspect run configuration, compute the baseline run, and generate a reusable Jython calibration script scaffold.

In [1]:
from pathlib import Path
import logging

import pandas as pd

logging.disable(logging.CRITICAL)


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "hms_commander").is_dir() and (candidate / "examples").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root()
WORK_ROOT = REPO_ROOT / "examples" / "working" / "clb238_guides"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
HMS_VERSION = "4.13"


from hms_commander import HmsExamples, HmsPrj

available_versions = HmsExamples.list_versions()
if HMS_VERSION not in available_versions:
    HMS_VERSION = available_versions[0]
HMS_EXE = HmsExamples.get_hms_exe(HMS_VERSION)


def init_sample_project(project_name, notebook_key):
    project_path = HmsExamples.extract_project(
        project_name,
        version=HMS_VERSION,
        output_path=WORK_ROOT / notebook_key,
        overwrite=True,
    )
    project = HmsPrj()
    project.initialize(project_path, hms_exe_path=HMS_EXE)
    return project, project_path

In [2]:
from hms_commander import HmsBasin, HmsCmdr, HmsJython

project, project_path = init_sample_project("castro", "26_calibration_validation")
run_name = "Current"
config = project.get_run_configuration(run_name)
observed_refs = project.get_observed_dss_paths()

observed_summary = pd.DataFrame([
    {"dss_file": path.name, "pathname": pathname, "record_type": pathname.split("/")[3] if len(pathname.split("/")) > 3 else ""}
    for path, pathname in observed_refs
])
assert not observed_summary.empty
observed_summary

,dss_file,pathname,record_type
0,castro.dss,/CASTRO VALLEY/FIRE DEPT./PRECIP-INC//10MIN/OBS/,PRECIP-INC
1,castro.dss,/CASTRO VALLEY/OUTLET/FLOW//10MIN/OBS/,FLOW


In [3]:
success = HmsCmdr.compute_run(
    run_name,
    hms_object=project,
    timeout=120,
    save_project=False,
    max_memory="2G",
)
assert success

baseline_outputs = pd.DataFrame([
    {"file": config["dss_file"], "exists": (project_path / config["dss_file"]).exists()},
    {"file": f"{run_name}.log", "exists": (project_path / f"{run_name}.log").exists()},
    {"file": f"{run_name}.out", "exists": (project_path / f"{run_name}.out").exists()},
])
assert baseline_outputs["exists"].all()
baseline_outputs

,file,exists
0,Current.dss,True
1,Current.log,True
2,Current.out,True


In [4]:
basin_path = Path(config["basin_file"])
subbasins = HmsBasin.get_subbasins(basin_path)
first_subbasin = subbasins.iloc[0]["name"]
calibration_parameters = {first_subbasin: {"CurveNumber": 75, "Lag": 30.0}}
calibration_script = HmsJython.generate_calibration_script(
    project_path,
    run_name,
    calibration_parameters,
    basin_name=config["basin_name"],
    hms_object=project,
)
script_check = pd.DataFrame([
    {"check": "contains CALIBRATION_PARAMS marker", "passed": "CALIBRATION_PARAMS" in calibration_script},
    {"check": "contains selected subbasin", "passed": first_subbasin in calibration_script},
    {"check": "contains run name", "passed": run_name in calibration_script},
])
assert script_check["passed"].all()
script_check

,check,passed
0,contains CALIBRATION_PARAMS marker,True
1,contains selected subbasin,True
2,contains run name,True


In [5]:
candidate_grid = pd.DataFrame([
    {"element": first_subbasin, "parameter": "CurveNumber", "candidate_value": value}
    for value in [70, 75, 80]
] + [
    {"element": first_subbasin, "parameter": "Lag", "candidate_value": value}
    for value in [20.0, 30.0, 40.0]
])
candidate_grid

,element,parameter,candidate_value
0,Subbasin-3,CurveNumber,70.0
1,Subbasin-3,CurveNumber,75.0
2,Subbasin-3,CurveNumber,80.0
3,Subbasin-3,Lag,20.0
4,Subbasin-3,Lag,30.0
5,Subbasin-3,Lag,40.0


## Coverage Notes

This notebook intentionally stops short of native HMS optimization trial management and DSS objective extraction. Those API gaps are tracked in CLB-290; this starter mirror still provides an executable baseline and a script-generation scaffold for manual calibration loops.